# Notebook 12 — Análisis Estadístico Comparativo: Modelo vs. Humanos

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 6 — Análisis estadístico de la tesis doctoral

---

## Estructura del notebook

| Sección | Contenido | Plan de análisis |
|---------|-----------|------------------|
| 1 | Setup, librerías, rutas | — |
| 2 | Carga y preparación de datos | — |
| 3 | Descriptivos del modelo | — |
| 4 | Descriptivos de los humanos | — |
| 5 | H1-1: Test binomial vs. azar (50%) | — |
| 6 | GLMM — precisión (variable binaria) | §8.5.1 |
| 7 | ANOVA 2×2×2×2 sobre TR + supuestos + LMM | §8.5.2 |
| 8 | Chi-cuadrado en ilusiones visuales | §8.5.3 |
| 9 | Kappa de Cohen + correlación de perfiles + correlación TR | §8.5.4 |
| 10 | Sesgo del modelo (tendencia ‘más cercano’) | — |
| 11 | Visualizaciones finales | — |
| 12 | Resumen y guardado de resultados | — |

## Fuentes de datos

- **Modelo:** `results/model_validation/model_responses_v2.csv` — NB10 (190 tareas × 3 repeticiones = 570 registros)
- **Humanos:** `results/human_responses/{P001,P002,P003}_tareas.csv` — App Quest 2 (190 tareas × 3 participantes = 570 registros)

## Nota sobre datos simulados

Si los archivos de participantes no están disponibles, el notebook genera datos sintéticos para verificar el pipeline. Los datos simulados se marcan explícitamente y se excluyen de la interpretación.

---

## Celda 1 — Librerías, rutas y configuración

In [ ]:
import subprocess, sys
for pkg in ['pingouin', 'statsmodels', 'scikit-learn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], capture_output=True)

from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import binomtest, chi2_contingency, shapiro, levene
from sklearn.metrics import cohen_kappa_score
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pingouin as pg
warnings.filterwarnings('ignore')

BASE_DIR    = '/content/drive/MyDrive/cognitive-depth-model'
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
MODEL_DIR   = os.path.join(RESULTS_DIR, 'model_validation')
HUMAN_DIR   = os.path.join(RESULTS_DIR, 'human_responses')
NB12_DIR    = os.path.join(RESULTS_DIR, 'nb12_analisis')
os.makedirs(NB12_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif',
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORES = {'modelo': '#1a5276', 'humano': '#e74c3c',
           'kitti':  '#2ecc71', 'ilusion': '#e67e22'}

print('✓ Librerías cargadas.')
print(f'  pingouin {pg.__version__}  |  statsmodels {sm.__version__}')
print(f'  Resultados en: {NB12_DIR}')

## Celda 2 — Carga y preparación de datos

In [ ]:
# 2.1 Datos del modelo (NB10)
ruta_modelo = os.path.join(MODEL_DIR, 'model_responses_v2.csv')
if not os.path.exists(ruta_modelo):
    ruta_modelo = os.path.join(MODEL_DIR, 'model_responses.csv')

df_modelo = pd.read_csv(ruta_modelo)
if 'prediccion' in df_modelo.columns:
    df_modelo = df_modelo.rename(columns={'prediccion':'respuesta','tiempo_ms':'tiempo_respuesta_ms'})
df_modelo['tipo_agente'] = 'modelo'
df_modelo['agente_id']   = 'modelo_rep' + df_modelo['repeticion'].astype(str)

print(f'✓ Datos del modelo: {len(df_modelo)} registros')
print(f'  Repeticiones: {sorted(df_modelo["repeticion"].unique())}')
print(f'  Accuracy global: {df_modelo["acierto"].mean():.4f} ({df_modelo["acierto"].mean()*100:.1f}%)')
print()

# 2.2 Datos humanos (app Quest 2)
PARTICIPANTES = ['P001', 'P002', 'P003']
dfs_humanos = []
participantes_disponibles = []

for pid in PARTICIPANTES:
    ruta = os.path.join(HUMAN_DIR, f'{pid}_tareas.csv')
    if os.path.exists(ruta):
        df_p = pd.read_csv(ruta)
        df_p['agente_id']   = pid
        df_p['tipo_agente'] = 'humano'
        if 'respuesta_humana' in df_p.columns:
            df_p = df_p.rename(columns={'respuesta_humana':'respuesta'})
        dfs_humanos.append(df_p)
        participantes_disponibles.append(pid)
        print(f'  ✓ {pid}: {len(df_p)} tareas | acc={df_p["acierto"].mean():.4f}')
    else:
        print(f'  ⚠ {pid}: no encontrado — se usarán datos simulados')

DATOS_SIMULADOS = len(participantes_disponibles) == 0

if DATOS_SIMULADOS:
    print()
    print('⚠ MODO SIMULACIÓN — NO interpretar resultados.')
    np.random.seed(42)
    df_val = pd.read_csv(os.path.join(BASE_DIR, 'data', 'splits', 'validation_set_final.csv'))
    for pid in PARTICIPANTES:
        df_p = df_val.copy()
        df_p['agente_id']   = pid
        df_p['tipo_agente'] = 'humano'
        df_p['es_piloto']   = False
        df_p['acierto']     = np.random.binomial(1, 0.72, len(df_p))
        df_p['respuesta']   = df_p.apply(
            lambda r: r['ground_truth'] if r['acierto']
            else ('mas_lejano' if r['ground_truth']=='mas_cercano' else 'mas_cercano'), axis=1)
        df_p['tiempo_respuesta_ms'] = np.random.lognormal(7.5, 0.4, len(df_p)).astype(int)
        dfs_humanos.append(df_p)
    participantes_disponibles = PARTICIPANTES

df_humanos = pd.concat(dfs_humanos, ignore_index=True)
print(f'✓ Datos humanos: {len(df_humanos)} registros | {len(participantes_disponibles)} participantes')

In [ ]:
# 2.3 Dataset unificado
COLS_COMUNES = ['agente_id','tipo_agente','numero_tarea','imagen_id',
                'dataset','tipo_tarea','nivel_disparidad',
                'presencia_distractores','ground_truth','respuesta',
                'acierto','tiempo_respuesta_ms']

for col in COLS_COMUNES:
    for df in [df_modelo, df_humanos]:
        if col not in df.columns:
            df[col] = np.nan

df_all = pd.concat([df_modelo[COLS_COMUNES], df_humanos[COLS_COMUNES]], ignore_index=True)
df_all['celda_factorial'] = (df_all['tipo_tarea'].str[:4].str.upper() + ' / ' +
                             df_all['nivel_disparidad'] + ' / ' +
                             df_all['presencia_distractores'].str[:3])

df_mod   = df_all[df_all['tipo_agente']=='modelo'].copy()
df_hum   = df_all[df_all['tipo_agente']=='humano'].copy()
df_ilus  = df_all[df_all['tipo_tarea']=='ilusion'].copy()
df_kitti = df_all[df_all['tipo_tarea']=='discriminacion'].copy()

print(f'✓ Dataset unificado: {len(df_all)} registros')
print(f'  Modelo: {len(df_mod)} | Humanos: {len(df_hum)}')
print(f'  KITTI: {len(df_kitti)} | Ilusiones: {len(df_ilus)}')
print()
print('N por celda factorial (modelo):')
print(df_mod.groupby(['tipo_tarea','nivel_disparidad','presencia_distractores'])['acierto']
      .count().to_string())

## Celda 3 — Descriptivos: Modelo

In [ ]:
print('=' * 65)
print('DESCRIPTIVOS DEL MODELO ARTIFICIAL')
print('=' * 65)

acc_global_mod = df_mod['acierto'].mean()
print(f'Accuracy global (3 reps): {acc_global_mod:.4f} ({acc_global_mod*100:.1f}%)')
print()
print('Por repetición:')
for r in sorted(df_mod['agente_id'].unique()):
    a = df_mod[df_mod['agente_id']==r]['acierto'].mean()
    print(f'  {r}: {a:.4f} ({a*100:.1f}%)')
print()
print('Por tipo de tarea:')
for t in ['discriminacion','ilusion']:
    a = df_mod[df_mod['tipo_tarea']==t]['acierto'].mean()
    n = (df_mod['tipo_tarea']==t).sum()
    print(f'  {t:20s}: {a:.4f} ({a*100:.1f}%)  N={n}')
print()
print('Por nivel de disparidad:')
for d in ['alta','baja']:
    a = df_mod[df_mod['nivel_disparidad']==d]['acierto'].mean()
    print(f'  {d:5s}: {a:.4f} ({a*100:.1f}%)')
print()
print('Por presencia de distractores:')
for dist in ['con_distractores','sin_distractores']:
    a = df_mod[df_mod['presencia_distractores']==dist]['acierto'].mean()
    print(f'  {dist:25s}: {a:.4f} ({a*100:.1f}%)')
print()
print('Accuracy por celda factorial 2x2x2 (modelo):')
print('-' * 65)
tabla_mod = df_mod.groupby(
    ['tipo_tarea','nivel_disparidad','presencia_distractores']
)['acierto'].agg(['mean','count']).reset_index()
tabla_mod.columns = ['Tarea','Disparidad','Distractores','Accuracy','N']
tabla_mod['Accuracy_%'] = (tabla_mod['Accuracy']*100).round(1)
print(tabla_mod[['Tarea','Disparidad','Distractores','Accuracy_%','N']].to_string(index=False))
print()
tr_mod = df_mod['tiempo_respuesta_ms'].dropna()
print(f'TR inferencia (ms): media={tr_mod.mean():.1f}  DE={tr_mod.std():.1f}  '
      f'min={tr_mod.min():.1f}  max={tr_mod.max():.1f}')

## Celda 4 — Descriptivos: Participantes humanos

In [ ]:
print('=' * 65)
print('DESCRIPTIVOS DE LOS PARTICIPANTES HUMANOS')
print('=' * 65)

acc_global_hum = df_hum['acierto'].mean()
print(f'Accuracy global: {acc_global_hum:.4f} ({acc_global_hum*100:.1f}%)')
print()
print('Por participante:')
for pid in sorted(df_hum['agente_id'].unique()):
    df_p = df_hum[df_hum['agente_id']==pid]
    a  = df_p['acierto'].mean()
    tr = df_p['tiempo_respuesta_ms'].dropna()
    print(f'  {pid}: acc={a:.4f} ({a*100:.1f}%)  TR_medio={tr.mean():.0f}ms  DE={tr.std():.0f}ms')
print()
print('Por tipo de tarea:')
for t in ['discriminacion','ilusion']:
    df_t = df_hum[df_hum['tipo_tarea']==t]
    a  = df_t['acierto'].mean()
    tr = df_t['tiempo_respuesta_ms'].dropna()
    print(f'  {t:20s}: acc={a:.4f}  TR_medio={tr.mean():.0f}ms')
print()
print('Accuracy por celda factorial 2x2x2 (humanos):')
print('-' * 65)
tabla_hum = df_hum.groupby(
    ['tipo_tarea','nivel_disparidad','presencia_distractores']
)['acierto'].agg(['mean','count','std']).reset_index()
tabla_hum.columns = ['Tarea','Disparidad','Distractores','Accuracy','N','DE']
tabla_hum['Accuracy_%'] = (tabla_hum['Accuracy']*100).round(1)
tabla_hum['DE_%']       = (tabla_hum['DE']*100).round(1)
print(tabla_hum[['Tarea','Disparidad','Distractores','Accuracy_%','DE_%','N']].to_string(index=False))
print()
tr_hum = df_hum['tiempo_respuesta_ms'].dropna()
print(f'TR humanos (ms): media={tr_hum.mean():.0f}  mediana={tr_hum.median():.0f}  '
      f'DE={tr_hum.std():.0f}  P25={tr_hum.quantile(0.25):.0f}  P75={tr_hum.quantile(0.75):.0f}')

## Celda 5 — H1-1: Desempeño superior al azar (test binomial)

In [ ]:
print('=' * 65)
print('H1-1: TEST BINOMIAL — ¿LOS AGENTES SUPERAN EL AZAR (50%)?')
print('=' * 65)
print('H0: p = 0.50 | H1: p > 0.50 (cola derecha, alpha=0.05)')
print()

def test_binomial(nombre, datos, alpha=0.05):
    n     = len(datos)
    n_ok  = int(datos.sum())
    p_obs = datos.mean()
    res   = binomtest(n_ok, n, p=0.5, alternative='greater')
    ic    = res.proportion_ci(confidence_level=0.95)
    sig   = '*** SIGNIFICATIVO' if res.pvalue < alpha else 'n.s.'
    print(f'{nombre}:')
    print(f'  N={n}  Aciertos={n_ok}  Accuracy={p_obs:.4f} ({p_obs*100:.1f}%)')
    print(f'  IC 95%: [{ic.low:.4f}, {ic.high:.4f}]')
    print(f'  p (cola derecha) = {res.pvalue:.6f}  → {sig}')
    print()
    return {'nombre':nombre,'n':n,'aciertos':n_ok,'accuracy':round(p_obs,4),
            'ic_low':round(ic.low,4),'ic_up':round(ic.high,4),
            'p_valor':round(res.pvalue,6),'significativo':res.pvalue<alpha}

resultados_binomial = []
resultados_binomial.append(test_binomial('Modelo — Global', df_mod['acierto']))
resultados_binomial.append(test_binomial('Modelo — KITTI', df_mod[df_mod['tipo_tarea']=='discriminacion']['acierto']))
resultados_binomial.append(test_binomial('Modelo — Ilusiones', df_mod[df_mod['tipo_tarea']=='ilusion']['acierto']))
resultados_binomial.append(test_binomial('Humanos — Global', df_hum['acierto']))
for pid in sorted(df_hum['agente_id'].unique()):
    resultados_binomial.append(test_binomial(f'Humanos — {pid}',
        df_hum[df_hum['agente_id']==pid]['acierto']))

df_binomial = pd.DataFrame(resultados_binomial)
df_binomial.to_csv(os.path.join(NB12_DIR, 'resultados_binomial.csv'), index=False)
print('✓ Test binomial guardado.')

## Celda 6 — GLMM: precisión (variable binaria, enlace logit)

> **Nota metodológica:** Con 3 participantes, los efectos aleatorios serían inestables. Se implementa regresión logística con participante como efecto fijo (aproximación al GLMM). La significancia se evalúa mediante Likelihood Ratio Test (LRT).

In [ ]:
print('=' * 65)
print('GLMM — COMPARACIÓN DE PRECISIÓN (REGRESIÓN LOGÍSTICA MULTIFACTORIAL)')
print('=' * 65)

df_glm = df_all[['acierto','tipo_agente','tipo_tarea','nivel_disparidad',
                  'presencia_distractores','agente_id']].copy().dropna(subset=['acierto'])

df_glm['tipo_agente']      = pd.Categorical(df_glm['tipo_agente'],      ['modelo','humano'])
df_glm['tipo_tarea']       = pd.Categorical(df_glm['tipo_tarea'],       ['discriminacion','ilusion'])
df_glm['nivel_disparidad'] = pd.Categorical(df_glm['nivel_disparidad'], ['baja','alta'])
df_glm['presencia_distractores'] = pd.Categorical(
    df_glm['presencia_distractores'], ['sin_distractores','con_distractores'])

# Modelos
modelo_nulo = smf.logit('acierto ~ 1', data=df_glm).fit(disp=False)
formula_ppal = ('acierto ~ C(tipo_agente) + C(tipo_tarea) + '
                'C(nivel_disparidad) + C(presencia_distractores) + C(agente_id)')
modelo_ppal = smf.logit(formula_ppal, data=df_glm).fit(disp=False)
formula_full = ('acierto ~ C(tipo_agente)*C(tipo_tarea)*'
                'C(nivel_disparidad)*C(presencia_distractores) + C(agente_id)')
modelo_full = smf.logit(formula_full, data=df_glm).fit(disp=False, maxiter=300)

print('Convergencia:')
for lbl, m in [('Nulo', modelo_nulo),('Efectos ppales', modelo_ppal),('Completo', modelo_full)]:
    print(f'  {lbl:20s}: {"OK" if m.converged else "NO convergió"}')
print()

In [ ]:
from scipy.stats import chi2 as chi2_dist

def lrt(m_reducido, m_completo, nombre=''):
    lr    = -2 * (m_reducido.llf - m_completo.llf)
    df_d  = m_completo.df_model - m_reducido.df_model
    p     = chi2_dist.sf(lr, df_d)
    sig   = '***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'n.s.'))
    print(f'  {nombre:40s}  χ²({int(df_d)})={lr:.3f}  p={p:.4f}  {sig}')
    return {'test':nombre,'chi2':round(lr,3),'df':int(df_d),'p_valor':round(p,6),'sig':sig}

print('Likelihood Ratio Tests (LRT):')
print('-' * 65)
res_lrt = []
res_lrt.append(lrt(modelo_nulo, modelo_ppal, 'Efectos ppales vs. nulo'))
for factor in ['C(tipo_agente)','C(tipo_tarea)','C(nivel_disparidad)','C(presencia_distractores)']:
    f_sin = formula_ppal.replace(' + '+factor,'').replace(factor+' + ','')
    try:
        m_sin = smf.logit(f_sin, data=df_glm).fit(disp=False)
        res_lrt.append(lrt(m_sin, modelo_ppal, f'Efecto de {factor}'))
    except Exception:
        pass
res_lrt.append(lrt(modelo_ppal, modelo_full, 'Interacciones vs. ppales'))
print()
print('Signif.: *** p<.001  ** p<.01  * p<.05  n.s. p>.05')

In [ ]:
# Odds Ratio e IC 95%
print()
print('Odds Ratio (OR) con IC 95% — Efectos principales:')
print('-' * 65)
params = modelo_ppal.params
conf   = modelo_ppal.conf_int()
pvals  = modelo_ppal.pvalues
for idx in params.index:
    if 'agente_id' in idx or idx == 'Intercept': continue
    lbl = (idx.replace('C(tipo_agente)[T.humano]','Agente: humano vs. modelo')
              .replace('C(tipo_tarea)[T.ilusion]','Tarea: ilusión vs. KITTI')
              .replace('C(nivel_disparidad)[T.alta]','Disparidad: alta vs. baja')
              .replace('C(presencia_distractores)[T.con_distractores]','Distractores: con vs. sin'))
    OR  = np.exp(params[idx])
    low = np.exp(conf.loc[idx, 0])
    up  = np.exp(conf.loc[idx, 1])
    p   = pvals[idx]
    sig = '***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'n.s.'))
    print(f'  {lbl[:45]:45s} OR={OR:.3f} [{low:.3f}, {up:.3f}]  {sig}')

# Pseudo-R2 Nagelkerke
def pseudo_r2_nk(modelo):
    n    = modelo.nobs
    L0   = np.exp(modelo_nulo.llf)
    Lm   = np.exp(modelo.llf)
    r2cs = 1 - (L0/Lm)**(2/n)
    return round(r2cs / (1 - L0**(2/n)), 4)

r2_ppal = pseudo_r2_nk(modelo_ppal)
r2_completo = pseudo_r2_nk(modelo_full)
print(f'\nPseudo-R² Nagelkerke: efectos ppales={r2_ppal}  completo={r2_completo}')

pd.DataFrame(res_lrt).to_csv(os.path.join(NB12_DIR, 'glmm_lrt.csv'), index=False)
print('✓ GLMM guardado.')

## Celda 7 — ANOVA 2×2×2×2 sobre TR + supuestos + LMM

> El ANOVA factorial se aplica sobre los **TR humanos**. Los TR del modelo (tiempos de inferencia GPU) se reportan descriptivamente aparte.

In [ ]:
print('=' * 65)
print('ANOVA / LMM — TIEMPOS DE RESPUESTA HUMANOS')
print('=' * 65)

df_tr = df_hum[['agente_id','tipo_tarea','nivel_disparidad',
                 'presencia_distractores','acierto','tiempo_respuesta_ms']].copy().dropna()
df_tr = df_tr[df_tr['tiempo_respuesta_ms'] >= 200]
mu, sigma = df_tr['tiempo_respuesta_ms'].mean(), df_tr['tiempo_respuesta_ms'].std()
df_tr = df_tr[np.abs(df_tr['tiempo_respuesta_ms'] - mu) < 3*sigma]
print(f'N tras filtrado outliers: {len(df_tr)}')
print(f'TR: media={df_tr["tiempo_respuesta_ms"].mean():.0f}ms  DE={df_tr["tiempo_respuesta_ms"].std():.0f}ms')
print()

# Supuestos
print('Supuestos del ANOVA:')
normalidad_ok = True
for nombre, grupo in df_tr.groupby(['tipo_tarea','nivel_disparidad','presencia_distractores']):
    if len(grupo) >= 5:
        stat, p = shapiro(grupo['tiempo_respuesta_ms'])
        if p <= .05: normalidad_ok = False
        print(f'  {str(nombre):52s}  Shapiro p={p:.4f}  {"OK" if p>.05 else "VIOLA"}')
print()
grupos_l = [g['tiempo_respuesta_ms'].values for _, g in
            df_tr.groupby(['tipo_tarea','nivel_disparidad','presencia_distractores'])]
stat_l, p_l = levene(*grupos_l)
homo_ok = p_l > .05
print(f'Levene: F={stat_l:.3f}  p={p_l:.4f}  {"OK" if homo_ok else "VIOLA"}')
print()
USAR_LMM = not (normalidad_ok and homo_ok)

In [ ]:
if not USAR_LMM:
    print('ANOVA DE MEDIDAS REPETIDAS (rm_anova):')
    try:
        aov = pg.rm_anova(data=df_tr,
                          dv='tiempo_respuesta_ms',
                          within=['tipo_tarea','nivel_disparidad','presencia_distractores'],
                          subject='agente_id', detailed=True)
        cols_show = [c for c in ['Source','F','ddof1','ddof2','p-unc','np2'] if c in aov.columns]
        print(aov[cols_show].round(4).to_string(index=False))
        aov.to_csv(os.path.join(NB12_DIR, 'anova_tr.csv'), index=False)
    except Exception as e:
        print(f'  [Error rm_anova: {e}] → usando LMM')
        USAR_LMM = True

if USAR_LMM:
    print('MODELO LINEAL MIXTO (LMM — supuestos violados):')
    from statsmodels.formula.api import mixedlm
    f_lmm = ('tiempo_respuesta_ms ~ C(tipo_tarea)+C(nivel_disparidad)+'
             'C(presencia_distractores)+'
             'C(tipo_tarea):C(nivel_disparidad)+'
             'C(tipo_tarea):C(presencia_distractores)+'
             'C(nivel_disparidad):C(presencia_distractores)')
    try:
        lmm_fit = mixedlm(f_lmm, df_tr, groups=df_tr['agente_id']).fit(reml=True)
        print(lmm_fit.summary().tables[1])
        print()
        print('Cohen\'s d para efectos principales:')
        for fac, niv in [('tipo_tarea',['discriminacion','ilusion']),
                          ('nivel_disparidad',['alta','baja']),
                          ('presencia_distractores',['con_distractores','sin_distractores'])]:
            g1 = df_tr[df_tr[fac]==niv[0]]['tiempo_respuesta_ms'].values
            g2 = df_tr[df_tr[fac]==niv[1]]['tiempo_respuesta_ms'].values
            sp = np.sqrt((g1.std()**2 + g2.std()**2)/2)
            d  = (g1.mean()-g2.mean())/sp if sp>0 else 0
            print(f'  {fac}: d = {d:.3f}')
    except Exception as e:
        print(f'  [Error LMM: {e}]')

## Celda 8 — Chi-cuadrado (χ²) en ilusiones visuales

In [ ]:
print('=' * 65)
print('CHI-CUADRADO — ERRORES EN ILUSIONES VISUALES')
print('=' * 65)
print('Objetivo: ¿humanos y modelo tienen niveles de error similares ante ilusiones?')
print()

df_ilus_c = df_ilus.copy()
df_ilus_c['error'] = 1 - df_ilus_c['acierto']

# Global
tabla_c = pd.crosstab(df_ilus_c['tipo_agente'], df_ilus_c['error'],
                      rownames=['Agente'], colnames=['Error'])
print('Tabla de contingencia:')
print(tabla_c)
print()

chi2_stat, p_chi2, df_chi2, _ = chi2_contingency(tabla_c, correction=False)
n_total  = tabla_c.values.sum()
v_cramer = np.sqrt(chi2_stat / (n_total * (min(tabla_c.shape)-1)))
sig_c = '***' if p_chi2<.001 else ('**' if p_chi2<.01 else ('*' if p_chi2<.05 else 'n.s.'))
tam_ef = 'peq.' if v_cramer<.1 else ('med.' if v_cramer<.3 else 'grande')
print(f'χ²({df_chi2}) = {chi2_stat:.4f}  p = {p_chi2:.6f}  {sig_c}')
print(f'V de Cramér = {v_cramer:.4f}  (efecto {tam_ef})')
print()

# Por condición
print('Chi-cuadrado por condición (ilusiones):')
print('-' * 65)
res_chi2_list = []
for (disp, dist), grupo in df_ilus_c.groupby(['nivel_disparidad','presencia_distractores']):
    tabla = pd.crosstab(grupo['tipo_agente'], grupo['error'])
    if tabla.shape == (2, 2):
        c2, p2, df2, _ = chi2_contingency(tabla, correction=False)
        vc = np.sqrt(c2 / (tabla.values.sum()))
        sig2 = '***' if p2<.001 else ('**' if p2<.01 else ('*' if p2<.05 else 'n.s.'))
        print(f'  Disp={disp:5s} / Dist={dist:20s}: '
              f'χ²({df2})={c2:.3f}  p={p2:.4f}  V={vc:.3f}  {sig2}')
        res_chi2_list.append({'disparidad':disp,'distractores':dist,'chi2':round(c2,4),
                               'df':df2,'p_valor':round(p2,6),'V_cramer':round(vc,4),'sig':sig2})

pd.DataFrame(res_chi2_list).to_csv(os.path.join(NB12_DIR, 'chi2_ilusiones.csv'), index=False)
print()
print('✓ Chi-cuadrado guardado.')

## Celda 9 — Correspondencia funcional humano–modelo (§8.5.4)

In [ ]:
print('=' * 65)
print('9.1 KAPPA DE COHEN — CONCORDANCIA ENSAYO POR ENSAYO')
print('=' * 65)

# Voto de mayoría del modelo por tarea (3 reps)
voto_modelo = df_mod.groupby('numero_tarea')['acierto'].mean().reset_index()
voto_modelo['bin'] = (voto_modelo['acierto'] >= 0.5).astype(int)

# Mayoría de acierto humano por tarea
voto_humano = df_hum.groupby('numero_tarea')['acierto'].mean().reset_index()
voto_humano['bin'] = (voto_humano['acierto'] >= 0.5).astype(int)

df_kappa = voto_modelo.merge(voto_humano, on='numero_tarea', suffixes=('_mod','_hum'))
kappa_global = cohen_kappa_score(df_kappa['bin_mod'], df_kappa['bin_hum'])

def interpretar_kappa(k):
    if k < .20:  return 'Concordancia pobre'
    elif k < .40: return 'Concordancia débil'
    elif k < .60: return 'Concordancia moderada'
    elif k < .80: return 'Concordancia sustancial'
    else:         return 'Concordancia casi perfecta'

print(f'κ global (190 tareas): {kappa_global:.4f}  → {interpretar_kappa(kappa_global)}')
print(f'(Criterios: Landis & Koch, 1977)')
print()

print('κ por tipo de tarea:')
res_kappa_list = [{'condicion':'Global','kappa':round(kappa_global,4),
                   'interpretacion':interpretar_kappa(kappa_global)}]
for tarea in ['discriminacion','ilusion']:
    idx_t = df_all[df_all['tipo_tarea']==tarea]['numero_tarea'].unique()
    df_kt = df_kappa[df_kappa['numero_tarea'].isin(idx_t)]
    if len(df_kt) >= 4:
        k_t = cohen_kappa_score(df_kt['bin_mod'], df_kt['bin_hum'])
        print(f'  {tarea:20s}: κ = {k_t:.4f}  → {interpretar_kappa(k_t)}')
        res_kappa_list.append({'condicion':tarea,'kappa':round(k_t,4),
                                'interpretacion':interpretar_kappa(k_t)})

pd.DataFrame(res_kappa_list).to_csv(os.path.join(NB12_DIR, 'kappa_cohen.csv'), index=False)

In [ ]:
print()
print('=' * 65)
print('9.2 CORRELACIÓN DE PERFILES DE PRECISIÓN (8 celdas factoriales)')
print('=' * 65)

acc_por_celda = df_all.groupby(
    ['tipo_agente','tipo_tarea','nivel_disparidad','presencia_distractores']
)['acierto'].mean().reset_index()

perfil_mod = acc_por_celda[acc_por_celda['tipo_agente']=='modelo']['acierto'].values
perfil_hum = acc_por_celda[acc_por_celda['tipo_agente']=='humano']['acierto'].values

_, p_nm = shapiro(perfil_mod)
_, p_nh = shapiro(perfil_hum)
if (p_nm > .05) and (p_nh > .05):
    r_perfil, p_perfil = stats.pearsonr(perfil_mod, perfil_hum)
    metodo = 'Pearson'
else:
    r_perfil, p_perfil = stats.spearmanr(perfil_mod, perfil_hum)
    metodo = 'Spearman'

sig_p = '***' if p_perfil<.001 else ('**' if p_perfil<.01 else ('*' if p_perfil<.05 else 'n.s.'))
print(f'r ({metodo}) = {r_perfil:.4f}  p = {p_perfil:.4f}  {sig_p}')
interp_r = 'alta' if abs(r_perfil)>.5 else ('moderada' if abs(r_perfil)>.3 else 'baja')
print(f'Interpretación: correlación {interp_r} — '
      f'{"condiciones difíciles para humanos también lo son para el modelo" if abs(r_perfil)>.5 else "sensibilidades divergentes"}')
print()

print('Perfiles por celda factorial:')
print(f'  {"Celda":45s} {"Modelo":>8} {"Humanos":>8}')
print('-' * 65)
for _, row in acc_por_celda[acc_por_celda['tipo_agente']=='modelo'].iterrows():
    etq  = f"{row['tipo_tarea'][:4].upper()} / {row['nivel_disparidad'][:4]} / {row['presencia_distractores'][:3]}"
    hval = acc_por_celda[(acc_por_celda['tipo_agente']=='humano') &
                          (acc_por_celda['tipo_tarea']==row['tipo_tarea']) &
                          (acc_por_celda['nivel_disparidad']==row['nivel_disparidad']) &
                          (acc_por_celda['presencia_distractores']==row['presencia_distractores'])]['acierto']
    hv   = hval.values[0] if len(hval) > 0 else np.nan
    print(f'  {etq:45s} {row["acierto"]:>8.4f} {hv:>8.4f}')

In [ ]:
print()
print('=' * 65)
print('9.3 CORRELACIÓN DE TIEMPOS DE RESPUESTA POR CONDICIÓN')
print('=' * 65)

tr_por_celda = df_all.groupby(
    ['tipo_agente','tipo_tarea','nivel_disparidad','presencia_distractores']
)['tiempo_respuesta_ms'].mean().reset_index()

tr_mod_celda = tr_por_celda[tr_por_celda['tipo_agente']=='modelo']['tiempo_respuesta_ms'].values
tr_hum_celda = tr_por_celda[tr_por_celda['tipo_agente']=='humano']['tiempo_respuesta_ms'].values
mask_tr = ~(np.isnan(tr_mod_celda) | np.isnan(tr_hum_celda))
tr_mod_c = tr_mod_celda[mask_tr]
tr_hum_c = tr_hum_celda[mask_tr]

if len(tr_mod_c) >= 3:
    _, p_nm = shapiro(tr_mod_c)
    _, p_nh = shapiro(tr_hum_c)
    if (p_nm > .05) and (p_nh > .05):
        r_tr, p_tr = stats.pearsonr(tr_mod_c, tr_hum_c)
        met_tr = 'Pearson'
    else:
        r_tr, p_tr = stats.spearmanr(tr_mod_c, tr_hum_c)
        met_tr = 'Spearman'
    sig_tr = '***' if p_tr<.001 else ('**' if p_tr<.01 else ('*' if p_tr<.05 else 'n.s.'))
    print(f'Correlación de TR por condición ({met_tr}):')
    print(f'  r = {r_tr:.4f}  p = {p_tr:.4f}  {sig_tr}')
    print('  Nota: TR modelo = tiempo de inferencia GPU; TR humanos = tiempo cognitivo-motor.')
    print('  Comparación interpretativa, no de equivalencia directa.')
else:
    r_tr, p_tr, met_tr = np.nan, np.nan, 'N/A'
    print('  [Insuficientes celdas con datos de TR en ambos agentes]')

# Guardar resultados de correspondencia
_corresp = {
    'analisis':     ['Kappa_global', 'Correlacion_perfiles_precision', 'Correlacion_TR'],
    'estadistico':  [round(kappa_global, 4), round(r_perfil, 4),
                     round(r_tr, 4) if not np.isnan(r_tr) else None],
    'p_valor':      [None, round(p_perfil, 6),
                     round(p_tr, 6) if not np.isnan(p_tr) else None],
    'metodo':       ['Cohen_Kappa', metodo, met_tr],
}
res_corresp = pd.DataFrame(_corresp)
res_corresp.to_csv(os.path.join(NB12_DIR, 'correspondencia_funcional.csv'), index=False)
print()
print('\u2713 Análisis de correspondencia funcional guardados.')


## Celda 10 — Sesgo del modelo: tendencia ‘más cercano’

In [ ]:
print('=' * 65)
print('SESGO DEL MODELO \u2014 TENDENCIA DE RESPUESTA')
print('=' * 65)

resp_mod = df_mod['respuesta'].value_counts()
resp_hum = df_hum['respuesta'].value_counts()

for label, dist in [('Modelo', resp_mod), ('Humanos', resp_hum)]:
    total = dist.sum()
    print(f'{label}:')
    for resp_tipo in ['mas_cercano','mas_lejano']:
        n   = dist.get(resp_tipo, 0)
        pct = n / total * 100 if total > 0 else 0
        print(f'  {resp_tipo:15s}: n={n:4d}  ({pct:.1f}%)')
    print()

n_cercano_mod = resp_mod.get('mas_cercano', 0)
n_total_mod   = resp_mod.sum()
res_sesgo = binomtest(n_cercano_mod, n_total_mod, p=0.5, alternative='two-sided')
print(f'Test binomial de sesgo en el modelo:')
print(f'  Respuestas "más cercano": {n_cercano_mod}/{n_total_mod} ({n_cercano_mod/n_total_mod*100:.1f}%)')
print(f'  p-valor (bilateral): {res_sesgo.pvalue:.6f}')
if res_sesgo.pvalue < .05:
    print('  \u2192 Sesgo significativo: el modelo tiende a responder')
    print('    "más cercano" con mayor frecuencia de la esperada por azar.')
    print('    Interpretación teórica: consistente con la prioridad evolutiva')
    print('    de detección de objetos cercanos en el sistema visual biológico.')
else:
    print('  \u2192 Sin sesgo significativo en la distribución de respuestas.')

print()
print('Sesgo por dataset:')
for tarea, label in [('discriminacion','KITTI'), ('ilusion','Ilusiones')]:
    df_ds = df_mod[df_mod['tipo_tarea']==tarea]
    rc    = df_ds['respuesta'].value_counts()
    nc    = rc.get('mas_cercano', 0)
    nt    = rc.sum()
    print(f'  {label}: "más cercano" = {nc}/{nt} ({nc/nt*100:.1f}%) | '
          f'"más lejano" = {nt-nc}/{nt} ({(nt-nc)/nt*100:.1f}%)')

_sesgo_data = {
    'agente':         ['modelo', 'humanos'],
    'n_mas_cercano':  [n_cercano_mod, resp_hum.get('mas_cercano', 0)],
    'n_mas_lejano':   [resp_mod.get('mas_lejano', 0), resp_hum.get('mas_lejano', 0)],
    'pct_cercano':    [n_cercano_mod / n_total_mod,
                       resp_hum.get('mas_cercano', 0) / resp_hum.sum()],
}
res_sesgo_df = pd.DataFrame(_sesgo_data)
res_sesgo_df.to_csv(os.path.join(NB12_DIR, 'sesgo_respuesta.csv'), index=False)
print()
print('\u2713 Análisis de sesgo guardado.')


## Celda 11 — Visualizaciones finales

In [ ]:
fig = plt.figure(figsize=(16, 22))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.48, wspace=0.35)

# Plot 1: Accuracy por celda factorial (modelo vs humanos)
ax1 = fig.add_subplot(gs[0, :])
acc_comp = df_all.groupby(['tipo_agente','celda_factorial'])['acierto'].mean().reset_index()
celdas_u = acc_comp[acc_comp['tipo_agente']=='modelo'].sort_values('acierto')['celda_factorial'].values
x, w = np.arange(len(celdas_u)), 0.35
for agente, color, offset in [('modelo', COLORES['modelo'], -w/2), ('humano', COLORES['humano'], w/2)]:
    vals = [acc_comp[(acc_comp['tipo_agente']==agente)&(acc_comp['celda_factorial']==c)]['acierto'].values
            for c in celdas_u]
    vals = [v[0] if len(v)>0 else np.nan for v in vals]
    ax1.bar(x+offset, vals, w, label=agente.capitalize(), color=color, alpha=0.85)
ax1.axhline(0.5, color='gray', ls='--', lw=1.5, label='Azar=50%')
ax1.set_xticks(x)
ax1.set_xticklabels([c[:28] for c in celdas_u], rotation=30, ha='right', fontsize=8)
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1)
ax1.set_title('Accuracy por celda factorial 2×2×2: Modelo vs. Humanos', fontweight='bold')
ax1.legend(fontsize=9)

# Plot 2: Correlación perfiles
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(perfil_mod, perfil_hum, s=90, color=COLORES['modelo'], zorder=3)
if len(perfil_mod) > 1:
    z = np.polyfit(perfil_mod, perfil_hum, 1)
    xx = np.linspace(min(perfil_mod)-.05, max(perfil_mod)+.05, 50)
    ax2.plot(xx, np.poly1d(z)(xx), color=COLORES['humano'], lw=2, ls='--')
ax2.plot([0,1],[0,1], color='gray', lw=1, ls=':', label='Acuerdo perfecto')
ax2.set_xlabel('Accuracy Modelo')
ax2.set_ylabel('Accuracy Humanos')
ax2.set_title(f'Correlación de perfiles ({metodo})\nr={r_perfil:.3f}  p={p_perfil:.4f}', fontweight='bold')
ax2.set_xlim(0,1)
ax2.set_ylim(0,1)
ax2.legend(fontsize=8)

# Plot 3: Errores en ilusiones
ax3 = fig.add_subplot(gs[1, 1])
err_data = df_ilus.groupby('tipo_agente')['acierto'].agg(
    aciertos='sum', errores=lambda x: (1-x).sum()).reset_index()
for i, agente in enumerate(['modelo','humano']):
    row = err_data[err_data['tipo_agente']==agente].iloc[0]
    ax3.bar(i, row['aciertos'], color='#2ecc71', alpha=0.85,
            label='Aciertos' if i==0 else '')
    ax3.bar(i, row['errores'], bottom=row['aciertos'], color='#e74c3c', alpha=0.85,
            label='Errores' if i==0 else '')
ax3.set_xticks([0,1])
ax3.set_xticklabels(['Modelo','Humanos'])
ax3.set_ylabel('N respuestas')
ax3.set_title(f'Aciertos y Errores en Ilusiones\n'
              f'χ²({df_chi2})={chi2_stat:.3f}  V={v_cramer:.3f}', fontweight='bold')
ax3.legend(fontsize=9)

# Plot 4: TR humanos por condición
ax4 = fig.add_subplot(gs[2, 0])
tr_c = df_hum.groupby('tipo_tarea')['tiempo_respuesta_ms'].agg(['mean','std']).reset_index()
bars_c = ax4.bar(tr_c['tipo_tarea'], tr_c['mean'], yerr=tr_c['std'], capsize=5,
                  color=[COLORES['kitti'],COLORES['ilusion']], alpha=0.85)
ax4.set_ylabel('TR medio (ms)')
ax4.set_title('TR Humanos por Tipo de Tarea\n(media ± DE)', fontweight='bold')

# Plot 5: Sesgo de respuesta
ax5 = fig.add_subplot(gs[2, 1])
for i, (ag, df_ag) in enumerate([('Modelo', df_mod), ('Humanos', df_hum)]):
    rc = df_ag['respuesta'].value_counts(normalize=True)
    ax5.bar(i-.2, rc.get('mas_cercano',0), .35, label='Más cercano' if i==0 else '',
            color='#2980b9', alpha=0.85)
    ax5.bar(i+.2, rc.get('mas_lejano', 0), .35, label='Más lejano'  if i==0 else '',
            color='#e67e22', alpha=0.85)
ax5.set_xticks([0,1])
ax5.set_xticklabels(['Modelo','Humanos'])
ax5.axhline(0.5, color='gray', ls='--', lw=1.5)
ax5.set_ylabel('Proporción')
ax5.set_ylim(0,1)
ax5.set_title(f'Sesgo de Respuesta\nκ={kappa_global:.3f} ({interpretar_kappa(kappa_global)[:12]})', fontweight='bold')
ax5.legend(fontsize=9)

# Plot 6: Heatmap Kappa por celda
ax6 = fig.add_subplot(gs[3, :])
kappa_celda_list = []
for (tt, nd, pd_), grp in df_all.groupby(['tipo_tarea','nivel_disparidad','presencia_distractores']):
    tareas_c = grp['numero_tarea'].unique()
    df_kc    = df_kappa[df_kappa['numero_tarea'].isin(tareas_c)]
    if len(df_kc) >= 4:
        k_c = cohen_kappa_score(df_kc['bin_mod'], df_kc['bin_hum'])
        kappa_celda_list.append({'tipo_tarea':tt,'nivel_disparidad':nd,'presencia_distractores':pd_,'kappa':k_c})
df_kc2 = pd.DataFrame(kappa_celda_list)
if not df_kc2.empty:
    pivot_k = df_kc2.pivot_table(values='kappa',
                                  index=['tipo_tarea','nivel_disparidad'],
                                  columns='presencia_distractores')
    sns.heatmap(pivot_k, ax=ax6, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0, vmin=-0.5, vmax=1, linewidths=0.5,
                cbar_kws={'label': 'κ (Kappa de Cohen)'})
    ax6.set_title('Kappa de Cohen por Celda Factorial 2×2×2', fontweight='bold')

plt.suptitle('Análisis Comparativo: Modelo Cognitivo vs. Participantes Humanos\n'
             'Percepción de Profundidad — Diseño factorial 2×2×2',
             fontsize=13, fontweight='bold', y=1.01)
ruta_fig = os.path.join(NB12_DIR, 'analisis_comparativo_completo.png')
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Figura guardada: {ruta_fig}')

## Celda 12 — Resumen ejecutivo y guardado final

In [ ]:
print('=' * 65)
print('RESUMEN EJECUTIVO \u2014 ANÁLISIS COMPARATIVO NB12')
print('=' * 65)
print()
if DATOS_SIMULADOS:
    print('\u26a0 ADVERTENCIA: Resultados basados en DATOS SIMULADOS.')
    print('  Ejecutar nuevamente con datos reales para resultados válidos.')
    print()

print(f'DATOS: {len(df_all)} registros totales')
print(f'  Modelo:   {len(df_mod)} (190 tareas x 3 repeticiones)')
print(f'  Humanos:  {len(df_hum)} (190 tareas x {len(participantes_disponibles)} participantes)')
print()

print('RESULTADOS PRINCIPALES:')
print()
print('  H1-1 \u2014 Test binomial:')
m_acc = df_mod['acierto'].mean()
h_acc = df_hum['acierto'].mean()
print(f'    Modelo global:  acc={m_acc:.4f}  '
      f'{chr(10003)+" supera el azar" if m_acc>0.5 else chr(10007)+" no supera el azar"}')
print(f'    Humanos global: acc={h_acc:.4f}  '
      f'{chr(10003)+" supera el azar" if h_acc>0.5 else chr(10007)+" no supera el azar"}')
print()
print(f'  GLMM: Pseudo-R\u00b2 Nagelkerke = {r2_completo:.4f}')
print(f'  Chi-cuadrado ilusiones: \u03c7\u00b2={chi2_stat:.3f}  p={p_chi2:.4f}  V={v_cramer:.3f}')
print(f'  Kappa de Cohen (global): \u03ba = {kappa_global:.4f}  \u2192 {interpretar_kappa(kappa_global)}')
print(f'  Correlación perfiles precisión: r = {r_perfil:.4f}  p = {p_perfil:.4f}')
print()

_resumen = {
    'fecha_analisis':            datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'datos_simulados':           DATOS_SIMULADOS,
    'n_registros_total':         len(df_all),
    'n_modelo':                  len(df_mod),
    'n_humanos':                 len(df_hum),
    'participantes':             participantes_disponibles,
    'acc_modelo_global':         round(df_mod['acierto'].mean(), 4),
    'acc_humanos_global':        round(df_hum['acierto'].mean(), 4),
    'acc_modelo_kitti':          round(df_mod[df_mod['tipo_tarea']=='discriminacion']['acierto'].mean(), 4),
    'acc_modelo_ilusiones':      round(df_mod[df_mod['tipo_tarea']=='ilusion']['acierto'].mean(), 4),
    'glmm_pseudo_r2_nagelkerke': r2_completo,
    'chi2_ilusiones':            round(chi2_stat, 4),
    'chi2_p_valor':              round(p_chi2, 6),
    'v_cramer':                  round(v_cramer, 4),
    'kappa_global':              round(kappa_global, 4),
    'kappa_interpretacion':      interpretar_kappa(kappa_global),
    'r_perfiles_precision':      round(r_perfil, 4),
    'p_perfiles_precision':      round(p_perfil, 6),
    'metodo_correlacion':        metodo,
}
with open(os.path.join(NB12_DIR, 'resumen_analisis_nb12.json'), 'w', encoding='utf-8') as f:
    json.dump(_resumen, f, ensure_ascii=False, indent=2)

print('Archivos generados en', NB12_DIR, ':')
for archivo in sorted(os.listdir(NB12_DIR)):
    ruta_a = os.path.join(NB12_DIR, archivo)
    tam    = os.path.getsize(ruta_a) / 1024
    print(f'  \u2192 {archivo} ({tam:.1f} KB)')
print()
print('\u2713 NB12 completado. Todos los resultados guardados en Drive.')
